# Token & Price Exploration

## Summary

1. Load BioRED and contemporary abstracts datasets.
2. Perform an initial LLM extraction on a small BioRED sample (`biored_train_sample`).
3. Calculate `biored_train_sample` input/output token counts using the OpenAI tokenizer (`tiktoken`).
4. Extrapolate estimated API input/output costs for full benchmark and contemporary corpora using LLM API pricing per MTOK.
5. Export the structured `response` JSON for subsequent parsing and analysis.

In [1]:
import os

import json
import pandas as pd
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path

In [2]:
INPUT_PRICE_PER_MTOK = 0.74
OUTPUT_PRICE_PER_MTOK = 0.74 * 6

# Input Price

In [3]:
encoding = tiktoken.encoding_for_model("gpt-5")

In [4]:
br_dev = pd.read_csv("../data/processed/biored/br_dev.csv")
br_test = pd.read_csv("../data/processed/biored/br_test.csv")
br_train = pd.read_csv("../data/processed/biored/br_train.csv")

In [5]:
biored_dfs = {
    "dev": br_dev,
    "test": br_test,
    "train": br_train
}

In [6]:
def summarise_tokens(df: pd.DataFrame, name: str, price_per_mtok: float) -> dict:
    total_tokens = df["input_tokens"].sum()
    return {
        "name": name,
        "mean_tokens": df["input_tokens"].mean(),
        "median_tokens": df["input_tokens"].median(),
        "max_tokens": df["input_tokens"].max(),
        "total_tokens": total_tokens,
        "total_cost": total_tokens / 1_000_000 * price_per_mtok,
    }

In [7]:
summary_rows = []

for name, df in biored_dfs.items():
    df["input_tokens"] = df["abstract"].apply(lambda x: len(encoding.encode(x)))
    summary_rows.append(summarise_tokens(df, name, INPUT_PRICE_PER_MTOK))

biored_summary_df = pd.DataFrame(summary_rows)
display(biored_summary_df.set_index("name"))

print(f"Total cost: £{biored_summary_df['total_cost'].sum():.2f}")

,mean_tokens,median_tokens,max_tokens,total_tokens,total_cost
name,,,,,
dev,366.620,363.5,658,36662,0.027130
test,361.050,370.0,586,36105,0.026718
train,348.885,351.0,789,139554,0.103270


Total cost: £0.16


In [8]:
contemporary_corpus = pd.read_csv("../data/processed/contemporary/contemporary_corpus.csv")

In [9]:
contemporary_corpus["input_tokens"] = contemporary_corpus["abstract"].apply(lambda x: len(encoding.encode(x)))

contemporary_summary_df = pd.DataFrame([
    summarise_tokens(contemporary_corpus, "contemporary", INPUT_PRICE_PER_MTOK)
])
display(contemporary_summary_df.set_index("name"))

print(f"Total cost: £{contemporary_summary_df['total_cost'].sum():.2f}")

,mean_tokens,median_tokens,max_tokens,total_tokens,total_cost
name,,,,,
contemporary,345.326241,344.0,1807,340837,0.252219


Total cost: £0.25


In [10]:
combined_total_cost = biored_summary_df["total_cost"].sum() + contemporary_summary_df["total_cost"].sum()
combined_total_tokens = biored_summary_df["total_tokens"].sum() + contemporary_summary_df["total_tokens"].sum()

print(f"Combined total input tokens (BioRED + contemporary): {combined_total_tokens:,}")
print(f"Combined total input cost (BioRED + contemporary): £{combined_total_cost:.2f}")

Combined total input tokens (BioRED + contemporary): 553,158
Combined total input cost (BioRED + contemporary): £0.41


# Output Price

In [11]:
load_dotenv()
print(os.getenv("OPEN_AI_TEST_KEY")[:15])

sk-proj-4WDSBIA


In [12]:
client = OpenAI(
    api_key=os.getenv("OPEN_AI_TEST_KEY")
)

In [13]:
# Load BioRED abstracts
biored_train = pd.read_csv("../data/processed/biored/br_train.csv")

### Sample BioRED

In [14]:
biored_train_sample = biored_train.sample(
    n=35,
    random_state=42
).copy()

### Initial downsampled BioRED extraction run

In [15]:
results = []

for index, row in biored_train_sample.iterrows():
    abstract = row["abstract"]

    prompt = f"""
    You are a biomedical information extraction system.

    Extract all biomedical entities and relationships from the following abstract.

    Return ONLY valid JSON.

    Schema:

    {{
        "entities": [
            {{
                "name": "",
                "type": ""
            }}
        ],
        "relationships": [
            {{
                "source": "",
                "relation": "",
                "target": ""
            }}
        ]
    }}

    Abstract:

    {abstract}
    """

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    results.append({
        "pmid": row["pmid"],
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
        "output": response.output_text
    })

In [24]:
results_df = pd.DataFrame(results)
results_df.head(3)

,pmid,input_tokens,output_tokens,total_tokens,output
0,19319147,430,1139,1569,"{\n ""entities"": [\n {\n ""name"": ""Warf..."
1,15686794,200,728,928,"{\n ""entities"": [\n {\n ""name"": ""Amio..."
2,19521089,328,1530,1858,"{\n ""entities"": [\n {\n ""name"": ""Park..."


### Export `response` as JSON

In [23]:
from pathlib import Path
import json
import re

def extract_json(text: str) -> dict:
    """Strip markdown code fences (```json ... ```) if present, then parse."""
    text = text.strip()
    # remove ```json ... ``` or ``` ... ``` wrappers
    match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    if match:
        text = match.group(1)
    return json.loads(text)

parsed_results = []
failed_rows = []

for _, row in results_df.iterrows():
    try:
        extracted = extract_json(row["output"])
        parsed_results.append({
            "pmid": int(row["pmid"]),
            "entities": extracted["entities"],
            "relationships": extracted["relationships"]
        })
    except (json.JSONDecodeError, KeyError) as e:
        failed_rows.append({"pmid": row["pmid"], "error": str(e), "raw": row["output"]})

print(f"Parsed {len(parsed_results)} / {len(results_df)} rows successfully")
if failed_rows:
    print(f"{len(failed_rows)} rows failed to parse, e.g. pmid={failed_rows[0]['pmid']}")

output_path = Path("../data/results/biored_sample_extractions.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(parsed_results, f, indent=4)

print(f"Saved {len(parsed_results)} extractions to {output_path}")

Parsed 35 / 35 rows successfully
Saved 35 extractions to ../data/results/biored_sample_extractions.json


In [ ]:
results_df = pd.DataFrame(results)

display(results_df.head())

results_df["output_tokens"].describe(
    percentiles=[0.5, 0.9, 0.95]
)

,pmid,input_tokens,output_tokens,total_tokens,output
0,19319147,430,946,1376,"{\n ""entities"": [\n {\n ""name"": ""Warf..."
1,15686794,200,785,985,"{\n ""entities"": [\n {\n ""name"": ""Amio..."
2,19521089,328,1599,1927,"{\n ""entities"": [\n {\n ""name"": ""Park..."
3,19918264,350,1544,1894,"{\n ""entities"": [\n {\n ""name"": ""Fibr..."
4,24743235,383,1626,2009,"{\n ""entities"": [\n {\n ""name"": ""hema..."


count      35.000000
mean     1598.514286
std       487.746883
min       785.000000
50%      1540.000000
90%      2274.800000
95%      2478.500000
max      3037.000000
Name: output_tokens, dtype: float64

In [ ]:
# --- Downsampled train sample (actual measured costs from the API responses) ---
actual_input_cost = results_df["input_tokens"].sum() / 1_000_000 * INPUT_PRICE_PER_MTOK
actual_output_cost = results_df["output_tokens"].sum() / 1_000_000 * OUTPUT_PRICE_PER_MTOK
actual_total_cost = actual_input_cost + actual_output_cost

train_downsampled = {
    "abstracts": len(results_df),
    "num_tokens": int(results_df["input_tokens"].sum()),
    "mean_input_tokens": round(results_df["input_tokens"].mean(), 1),
    "median_input_tokens": round(results_df["input_tokens"].median(), 1),
    "max_input_tokens": int(results_df["input_tokens"].max()),
    "mean_output_tokens": round(results_df["output_tokens"].mean(), 1),
    "median_output_tokens": round(results_df["output_tokens"].median(), 1),
    "max_output_tokens": int(results_df["output_tokens"].max()),
    "input_cost": round(actual_input_cost, 2),
    "output_cost": round(actual_output_cost, 2),
    "total_cost": round(actual_total_cost, 2),
}

# --- Full-scale extrapolated costs ---
mean_output_tokens = results_df["output_tokens"].mean()
mean_output_price_per_abstract = mean_output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK

train_mean_input_tokens = biored_summary_df.loc[biored_summary_df["name"] == "train", "mean_tokens"].iloc[0]
mean_input_price_per_abstract = train_mean_input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK
mean_total_price_per_abstract = mean_input_price_per_abstract + mean_output_price_per_abstract

full_scale_targets = {
    "train_full": len(biored_train),
    "test_full": len(br_test),
    "dev_full": len(br_dev),
    "contemporary_full": len(contemporary_corpus),
}

full_scale_token_totals = {
    "train_full": int(biored_summary_df.loc[biored_summary_df["name"] == "train", "total_tokens"].iloc[0]),
    "test_full": int(biored_summary_df.loc[biored_summary_df["name"] == "test", "total_tokens"].iloc[0]),
    "dev_full": int(biored_summary_df.loc[biored_summary_df["name"] == "dev", "total_tokens"].iloc[0]),
    "contemporary_full": int(contemporary_summary_df.loc[contemporary_summary_df["name"] == "contemporary", "total_tokens"].iloc[0]),
}

full_scale_results = {
    label: {
        "abstracts": n_abstracts,
        "num_tokens": full_scale_token_totals[label],
        "total_cost": round(mean_total_price_per_abstract * n_abstracts, 2),
    }
    for label, n_abstracts in full_scale_targets.items()
}

cost_summary = {"train_downsampled": train_downsampled, **full_scale_results}

In [ ]:
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "token_price_summary.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(cost_summary, f, indent=4)

print(f"Saved cost summary to {output_path}")

Saved cost summary to ../data/results/token_price_summary.json


In [ ]:
display_labels = {
    "train_downsampled": ("BioRED train sample", "Actual API run"),
    "train_full": ("BioRED train", "Extrapolated"),
    "dev_full": ("BioRED dev", "Extrapolated"),
    "test_full": ("BioRED test", "Extrapolated"),
    "contemporary_full": ("Contemporary", "Extrapolated"),
}

summary_table_df = pd.DataFrame([
    {
        "Dataset": display_name,
        "Abstracts": cost_summary[key]["abstracts"],
        "Number of Tokens": cost_summary[key]["num_tokens"],
        "Basis": basis,
        "Total (actual / extrapolated) cost": f"£{cost_summary[key]["total_cost"]:.2f}",
    }
    for key, (display_name, basis) in display_labels.items()
])

display(summary_table_df.set_index("Dataset"))

,Abstracts,Number of Tokens,Basis,Total (actual / extrapolated) cost
Dataset,,,,
BioRED train sample,35,14625,Actual API run,£0.26
BioRED train,400,139554,Extrapolated,£2.94
BioRED dev,100,36662,Extrapolated,£0.74
BioRED test,100,36105,Extrapolated,£0.74
Contemporary,987,340837,Extrapolated,£7.26


In [ ]:
summary_table_df.to_csv("../data/results/token_usage_summary.csv", index=False)